In [60]:
from datetime import datetime, timedelta, date
import os
import requests
import time
import pandas as pd
import holidays
from category_encoders import TargetEncoder
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import xgboost as xgb
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [61]:
def ophalenKijkcijferData(startDate, endDate):
  print(f"Ophalen kijkcijfer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
  kijkcijfersData = []
  #elke dag ophalen (startDate is huidige dag)
  while startDate <= endDate:
    datum = f"{startDate.year}-{startDate.month}-{startDate.day}"
    url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"

    try:
      response = requests.get(url)
      if response.status_code == 200:
        data = response.json()
        programmaLijst = data.get('hydra:member', [])
                        
        for programma in programmaLijst:
          try:
            kijkcijfersData.append({
              'dateDiff': programma.get('dateDiff'),
              'ranking': programma.get('ranking'),
              'description': programma.get('description'),
              'channel': programma.get('channel'),
              'startTime': programma.get('startTime'),
              'rLength': programma.get('rLength'),
              'rateInK': programma.get('rateInK'),
              'live': programma.get('live')
            })
                                   
          except Exception as e:
            print(f"error {datum}: {e}")         
      else:
        print(f"no data {datum}")
                        
    except Exception as e:
      print(f"error: {e}")
    
    startDate += timedelta(days=1)

  print("KijkcijferData opgehaald")
  df = pd.DataFrame(kijkcijfersData)

  return df



# Pipeline opstellen

In [62]:
def testCSV(csv):
    df = pd.read_csv(csv, delimiter=';')
    # Kolommen hernoemen
    df = df.rename(columns={
        'Programma': 'description',
        'Zender': 'channel',
        'Datum ': 'dateDiff',
        'Start ': 'startTime',
        'Duur': 'rLength'
    })
    # Datum en tijd samenvoegen tot datetime
    df['dateDiff'] = pd.to_datetime(df['dateDiff'], dayfirst=True)
    # Starttijd naar HH:MM:SS
    df['startTime'] = df['startTime'].str[:8]
    # Duur naar HH:MM:SS
    df['rLength'] = df['rLength'].apply(lambda x: str(pd.to_timedelta(x)))
    # Voeg dummy kolommen toe als nodig
    df['ranking'] = 0
    df['live'] = 0
    return df[['dateDiff', 'ranking', 'description', 'channel', 'startTime', 'rLength', 'live']]


In [63]:
def testing(csv):
    df = pd.read_csv(csv, delimiter=';')
    # Kolommen hernoemen
    df = df.rename(columns={
        'Programma': 'description',
        'Zender': 'channel',
        'Datum ': 'dateDiff',
        'Start ': 'startTime',
        'Duur': 'rLength',
        'kijkers': 'rateInK'
    })
    # Datum en tijd samenvoegen tot datetime
    df['dateDiff'] = pd.to_datetime(df['dateDiff'], dayfirst=True)
    # Starttijd naar HH:MM:SS
    df['startTime'] = df['startTime'].str[:8]
    # Duur naar HH:MM:SS
    df['rLength'] = df['rLength'].apply(lambda x: str(pd.to_timedelta(x)))
    # Voeg dummy kolommen toe als nodig
    df['ranking'] = 0
    df['live'] = 0
    return df[['dateDiff', 'ranking', 'description', 'channel', 'startTime', 'rLength', 'rateInK', 'live']]


## Ophalen data

In [5]:
def ophalenKijkcijferData(startDate, endDate):
  print(f"Ophalen kijkcijfer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
  kijkcijfersData = []
  #elke dag ophalen (startDate is huidige dag)
  while startDate <= endDate:
    datum = f"{startDate.year}-{startDate.month}-{startDate.day}"
    url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"

    try:
      response = requests.get(url)
      if response.status_code == 200:
        data = response.json()
        programmaLijst = data.get('hydra:member', [])
                        
        for programma in programmaLijst:
          try:
            kijkcijfersData.append({
              'dateDiff': programma.get('dateDiff'),
              'ranking': programma.get('ranking'),
              'description': programma.get('description'),
              'channel': programma.get('channel'),
              'startTime': programma.get('startTime'),
              'rLength': programma.get('rLength'),
              'rateInK': programma.get('rateInK'),
              'live': programma.get('live')
            })
                                   
          except Exception as e:
            print(f"error {datum}: {e}")         
      else:
        print(f"no data {datum}")
                        
    except Exception as e:
      print(f"error: {e}")
    
    startDate += timedelta(days=1)

  print("KijkcijferData opgehaald")
  df = pd.DataFrame(kijkcijfersData)

  return df

def ophalenWeerData(startDate, endDate):
    latitude = 51.05
    longitude = 3.7167
    today = datetime.today().date()

    hourly_vars = [
        "temperature_2m", "apparent_temperature", "weather_code", "precipitation",
        "rain", "snowfall", "cloud_cover", "windspeed_10m", "sunshine_duration"
    ]
    
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }

    def fetch_weather_data(api_url, start, end):
        params = common_params.copy()
        params.update({
            "start_date": start.strftime('%Y-%m-%d'),
            "end_date": end.strftime('%Y-%m-%d')
        })
        # print(f"Ophalen weer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
        response = requests.get(api_url, params=params)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df = pd.DataFrame({var: data.get(var, []) for var in hourly_vars})
            df["timestamp"] = pd.to_datetime(data.get("time", []))
            if not df.empty:
                df["hour"] = df["timestamp"].dt.hour
                df["day_of_week"] = df["timestamp"].dt.dayofweek
                df["month"] = df["timestamp"].dt.month
                df["year"] = df["timestamp"].dt.year
            return df
        else:
            print(f"Fout bij ophalen data: {response.status_code}")
            print(response.text)
            return pd.DataFrame()

    dataframes = []

    # Historische data
    if startDate.date() < today:
        print("Ophalen historische data")
        hist_end = min(endDate.date(), today - timedelta(days=1))
        dataframes.append(fetch_weather_data(
            "https://archive-api.open-meteo.com/v1/archive",
            startDate, datetime.combine(hist_end, datetime.min.time())
        ))

    # Forecast data
    if endDate.date() >= today:
        print("Ophalen forecast data")
        forecast_start = max(endDate, datetime.combine(today, datetime.min.time()))
        print(forecast_start)
        dataframes.append(fetch_weather_data(
            "https://api.open-meteo.com/v1/forecast",
            forecast_start, endDate
        ))

    if dataframes:
        print("Weerdata opgehaald")
        return pd.concat(dataframes).sort_values("timestamp").reset_index(drop=True)
    else:
        return pd.DataFrame()


In [49]:
teVoorspellen = testing('./data csv/TEST_kijkcijfers.csv')

teVoorspellen['dateDiff'] = pd.to_datetime(teVoorspellen['dateDiff'])

end_date = teVoorspellen['dateDiff'].max()
start_date = end_date - timedelta(weeks=3)
histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
histWeerdata = ophalenWeerData(start_date, end_date)
histWeerdata

Ophalen kijkcijfer-data van 2025-4-25 tot 2025-5-15
KijkcijferData opgehaald
Ophalen historische data
Ophalen forecast data
2025-05-16 00:00:00
Weerdata opgehaald


,temperature_2m,apparent_temperature,weather_code,precipitation,rain,snowfall,cloud_cover,windspeed_10m,sunshine_duration,timestamp,hour,day_of_week,month,year
0,10.0,8.0,3.0,0.0,0.0,0.0,100.0,11.5,0.0,2025-04-25 00:00:00,0,4,4,2025
1,10.3,8.3,51.0,0.1,0.1,0.0,100.0,11.0,0.0,2025-04-25 01:00:00,1,4,4,2025
2,10.0,8.0,51.0,0.1,0.1,0.0,100.0,10.7,0.0,2025-04-25 02:00:00,2,4,4,2025
3,10.3,8.7,2.0,0.0,0.0,0.0,73.0,8.8,0.0,2025-04-25 03:00:00,3,4,4,2025
4,10.0,8.3,1.0,0.0,0.0,0.0,22.0,8.6,0.0,2025-04-25 04:00:00,4,4,4,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,14.9,11.0,0.0,0.0,0.0,0.0,0.0,20.5,3600.0,2025-05-16 19:00:00,19,4,5,2025
500,13.9,10.2,3.0,0.0,0.0,0.0,98.0,18.7,3600.0,2025-05-16 20:00:00,20,4,5,2025
501,13.3,9.9,3.0,0.0,0.0,0.0,100.0,16.9,0.0,2025-05-16 21:00:00,21,4,5,2025
502,12.8,9.5,3.0,0.0,0.0,0.0,100.0,17.3,0.0,2025-05-16 22:00:00,22,4,5,2025


In [50]:
teVoorspellen

,dateDiff,ranking,description,channel,startTime,rLength,rateInK,live
0,2025-05-16,0,THUIS,VRT 1,20:17:52,0 days 00:24:30,893.239,0
1,2025-05-16,0,HET 7 UUR-JOURNAAL,VRT 1,19:00:04,0 days 00:43:18,823.019,0
2,2025-05-16,0,MAN BIJT HOND,VRT 1,19:46:05,0 days 00:23:30,997.337,0
3,2025-05-16,0,BLOKKEN,VRT 1,18:29:07,0 days 00:28:30,501.293,0
4,2025-05-16,0,NIEUWS 19U VTM,VTM,18:59:48,0 days 00:53:21,527.109,0
5,2025-05-16,0,DE DAG VAN VANDAAG,VRT 1,21:37:08,0 days 00:53:26,620.036,0
6,2025-05-16,0,FAMILIE,VTM,20:05:28,0 days 00:25:44,521.098,0
7,2025-05-16,0,HUIS GEMAAKT,VTM,20:41:26,0 days 01:05:45,428.309,0
8,2025-05-16,0,GELUKKIG GESCHEIDEN,VRT 1,20:44:02,0 days 00:50:35,650.230,0
9,2025-05-16,0,HET 1 UUR-JOURNAAL,VRT 1,13:00:04,0 days 00:27:33,398.209,0


## Cleaning data

In [8]:
def cleanKijkcijferData(df):

    # Zet 'Kijkers' kolom, als 'rateInK' bestaat
    if 'rateInK' in df.columns:
        df['Kijkers'] = (
            df['rateInK']
            .dropna()
            .astype(str)
            .str.replace('.', '', regex=False)
            .astype(int)
        )
    else:
        df['Kijkers'] = None

    # rLength aanpassen
    df['rLength'] = df['rLength'].astype(str).apply(
    lambda x: x[-8:] if 'days' in x else x.zfill(8)
    )

    # Tijd aanpassen
    tijd_regex = r'^\d{2}:\d{2}:\d{2}$'
    # Omzetten naar datetime
    df['date'] = pd.to_datetime(df['dateDiff']).dt.date
    # Filter rijen met formaat
    df = df[df['startTime'].str.match(tijd_regex, na=False) & df['rLength'].str.match(tijd_regex, na=False)].copy()
    
    # Afleveringlengte naar seconden omzetten 
    df['Lengte_sec'] = pd.to_timedelta(df['rLength']).dt.total_seconds().astype(int)

    # Uren met 24+
    def time_cor(rij):
        tijdArr = rij['startTime'].split(':')
        if int(tijdArr[0]) >= 24:
            tijdArr[0] = str(int(tijdArr[0]) - 24).zfill(2)
            rij['date'] += timedelta(days=1)
        rij['startTime'] = ':'.join(tijdArr)
        return rij
    
    df = df.apply(time_cor, axis=1)

    # 1 kolom voor beide data
    df['FullDate'] = pd.to_datetime(df['date'].astype(str) 
                                    + " " + df['startTime'].astype(str))
    
    # Hour en minute voor join later on
    df['hour'] = pd.to_datetime(df['startTime'], format='%H:%M:%S').dt.hour
    df['minute'] = 0

    # Kolommen verwijderen die niet nodig meer zijn, als ze bestaan
    columns_to_drop = ['startTime', 'rLength', 'rateInK', 'ranking', 'live']
    df.drop([col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)


    # De nieuwe dataframe
    df = df[['FullDate', 'date', 'hour', 'minute', 'channel', 'description', 'Lengte_sec', 'Kijkers']]

    # Hernoemen kolommen
    df.rename(columns={'description': 'Programma', 'channel': 'Kanaal'}, inplace=True)

    return df


def cleanWeerData(df):
  weerData = df
  weerData['timestamp'] = pd.to_datetime(weerData['timestamp'])
  #naar zelfde formaat als kijkcijfer datum
  weerData['datetime'] = weerData['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

  #hour voor join later on
  weerData['hour'] = pd.to_datetime(weerData['datetime']).dt.hour
  weerData['minute'] = pd.to_datetime(weerData['datetime']).dt.minute
  weerData['date'] = pd.to_datetime(weerData['datetime']).dt.date

  #verwijder kolom
  weerData = weerData.drop(columns=['timestamp'])

  weerData = weerData[['datetime', 'date' ,'hour', 'minute', 'temperature_2m', 'apparent_temperature', 
                            'rain', 'snowfall', 'weather_code', 'cloud_cover', 
                            'windspeed_10m', 'sunshine_duration']]

  #hernoemen kolommen
  weerData.rename(columns={'temperature_2m':'Temperatuur', 'apparent_temperature':'Gevoelstemp', 'windspeed_10m': 'Windsnelheid', 'rain':'Regen', 'snowfall': 'Sneeuw', 'weather_code':'Weercode', 'cloud_cover':'Bewolking', 'sunshine_duration':'Zonnenschijn'}, inplace=True)

  return weerData

def mergen(kijkcijfers, weer):
  kijkcijfersWeer = pd.merge(kijkcijfers, weer, on=['date', 'hour'], how='left')
  kijkcijfersWeer = kijkcijfersWeer[['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec', 'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  kijkcijfersWeer.dropna(inplace=True)
  return kijkcijfersWeer

In [9]:
histWeerdataClean = cleanWeerData(histWeerdata)
histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
# Merge historische kijkcijfers en weerdata
histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
histKijkcijfersWeer.dropna(inplace=True)
# te voorspellen data en weerdata mergen
teVoorspellenClean = cleanKijkcijferData(teVoorspellen)
teVoorspellenClean = teVoorspellenClean.drop(columns=['minute'])
teVoorspellenData = pd.merge(teVoorspellenClean, histWeerdataClean, on=['date', 'hour'], how='left')
teVoorspellenData = teVoorspellenData.drop(columns=['datetime', 'minute'])
#print("Aantal rijen in histKijkcijferWeerDf:", len(histKijkcijfersWeer))
#print(histKijkcijfersWeer.head()) 
teVoorspellenData

,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn
0,2025-05-16 20:17:52,2025-05-16,20,VRT 1,THUIS,1470,893239,14.0,10.4,0.0,0.0,3.0,97.0,18.7,3600.0
1,2025-05-16 19:00:04,2025-05-16,19,VRT 1,HET 7 UUR-JOURNAAL,2598,823019,14.9,10.9,0.0,0.0,0.0,17.0,20.9,3600.0
2,2025-05-16 19:46:05,2025-05-16,19,VRT 1,MAN BIJT HOND,1410,997337,14.9,10.9,0.0,0.0,0.0,17.0,20.9,3600.0
3,2025-05-16 18:29:07,2025-05-16,18,VRT 1,BLOKKEN,1710,501293,15.7,11.7,0.0,0.0,0.0,0.0,21.2,3600.0
4,2025-05-16 18:59:48,2025-05-16,18,VTM,NIEUWS 19U VTM,3201,527109,15.7,11.7,0.0,0.0,0.0,0.0,21.2,3600.0
5,2025-05-16 21:37:08,2025-05-16,21,VRT 1,DE DAG VAN VANDAAG,3206,620036,13.3,10.0,0.0,0.0,3.0,100.0,16.9,0.0
6,2025-05-16 20:05:28,2025-05-16,20,VTM,FAMILIE,1544,521098,14.0,10.4,0.0,0.0,3.0,97.0,18.7,3600.0
7,2025-05-16 20:41:26,2025-05-16,20,VTM,HUIS GEMAAKT,3945,428309,14.0,10.4,0.0,0.0,3.0,97.0,18.7,3600.0
8,2025-05-16 20:44:02,2025-05-16,20,VRT 1,GELUKKIG GESCHEIDEN,3035,65023,14.0,10.4,0.0,0.0,3.0,97.0,18.7,3600.0
9,2025-05-16 13:00:04,2025-05-16,13,VRT 1,HET 1 UUR-JOURNAAL,1653,398209,18.8,15.7,0.0,0.0,0.0,0.0,18.4,3600.0


## OneHotEncoding

In [10]:
def oneHot(df):
  with open('./models/oneHotEncoder.pkl', 'rb') as oneHotFile:
    oneHotEnc = pickle.load(oneHotFile)

  lageKard = df[[ 'hour','Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen']]
  dfOneHot = oneHotEnc.transform(lageKard)

  oneHotOutp = pd.DataFrame(dfOneHot.toarray(), 
                            columns=oneHotEnc.get_feature_names_out(), 
                            index=lageKard.index)

  df = df.drop(columns=['hour', 'Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen'])
  df = pd.concat([df, oneHotOutp], axis = 1)
  return df

## TargetEncoding

In [11]:
def target(df):
  #target encoding voor medium kardinaliteiten
  with open('./models/oneHotTarget.pkl', 'rb') as f:
    targetEnc = pickle.load(f)
  medKardinaliteit = df[['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  #verdere feature engineering op vorig model
  target = targetEnc.transform(medKardinaliteit)
  df = df.drop(columns=['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn'])
  f = pd.concat([df, target], axis=1)

  return f


## Feature engineering

In [12]:
def tijdFeatures(df):
    cleanData = df
    cleanData['date'] = pd.to_datetime(cleanData['date'])
    #feestdagen
    feestdagen = holidays.BE()
    cleanData['isFeestdag'] = cleanData['date'].apply(lambda x: 1 if x in feestdagen else 0)
    #dag van de week
    cleanData['Weekdag'] = cleanData['date'].dt.weekday
    #weekend
    cleanData['isWeekend'] = cleanData['Weekdag'].apply(lambda x: 1 if x >= 5 else 0)
    #seizoenen
    cleanData['Seizoen'] = cleanData['date'].apply(seizoenFinder)

    return cleanData

#seizoen
def seizoenFinder(datum):
    inputDatum = datum.date()
    Y = inputDatum.year
    seizoenen = {
        'lente': (date(Y, 3, 20), date(Y, 6, 20)),
        'zomer': (date(Y, 6, 21), date(Y, 9, 22)),
        'herfst':   (date(Y, 9, 23), date(Y, 12, 20)),
        'winter': (date(Y, 12, 21), date(Y + 1, 3, 19)),
    }

    for seizoen, (start, end) in seizoenen.items():
        if start <= inputDatum <= end:
            return seizoen
    return 'winter'

def createLag(df, n):
  for i in range(1,n+1):
    df[f'KijkersLag{i}'] = df.sort_values('FullDate').groupby('Programma')['Kijkers'].shift(i).ffill()
  return df

def lagFeatures(predDf, histKijkcijferWeerDf):
  predDf['Kijkers'] = np.nan

  for i in range(1, 4):
     predDf[f'KijkersLag{i}'] = predDf.sort_values('FullDate').groupby(['Programma'])['Kijkers'].shift(i)
     predDf[f'KijkersLag{i}'] = predDf[f'KijkersLag{i}'].fillna(predDf.groupby(['Programma'])['Kijkers'].transform('mean'))
     
  predDf = pd.concat([histKijkcijferWeerDf, predDf], ignore_index=True)
  return predDf   


## Pipeline

In [56]:
def voorbereiding(toPredictData, histKijkcijfersWeer):
    # Vervang spaties door underscores in beide DataFrames
    toPredictData = toPredictData.copy()
    histKijkcijfersWeer = histKijkcijfersWeer.copy()
    toPredictData['Kanaal'] = toPredictData['Kanaal'].str.replace(' ', '_')
    histKijkcijfersWeer['Kanaal'] = histKijkcijfersWeer['Kanaal'].str.replace(' ', '_')
    # tijd features toevoegen
    tijdFeatures(toPredictData)
    tijdFeatures(histKijkcijfersWeer)

    print(histKijkcijfersWeer['Kanaal'].unique())


    # lag features toevoegen
    pred_hist_df = lagFeatures(toPredictData, histKijkcijfersWeer)

    # te voorspellen data er terug uithalen
    toPredictData = pred_hist_df[pred_hist_df['Kijkers'].isnull()]

    # One hot encoding
    toPredictData = oneHot(toPredictData)
    # Target encoding
    targetOneHotEnc = target(toPredictData)
    print(targetOneHotEnc.columns)

    if 'Kijkers' in targetOneHotEnc.columns:
        targetOneHotEnc = targetOneHotEnc.drop(columns=['Kijkers'])

    # numeric columns selecteren
    toPredictNumeric = targetOneHotEnc.select_dtypes(include=[np.number])
    #print(toPredictNumeric.columns)
    
    return toPredictNumeric

In [48]:
data = voorbereiding(teVoorspellenData, histKijkcijfersWeer)
data.columns

Index(['FullDate', 'Kijkers', 'Sneeuw', 'Weercode', 'isWeekend', 'KijkersLag1',
       'KijkersLag2', 'KijkersLag3', 'hour_0', 'hour_1', 'hour_2', 'hour_6',
       'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12',
       'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18',
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'Kanaal_AB3',
       'Kanaal_CANVAS', 'Kanaal_CAZ', 'Kanaal_Canvas',
       'Kanaal_DAZN_PRO_LEAGUE_1_(NL)', 'Kanaal_EEN',
       'Kanaal_ELEVEN_PRO_LEAGUE_1_NL', 'Kanaal_EUROSPORT_1_(NL)',
       'Kanaal_KETNET', 'Kanaal_LA_UNE', 'Kanaal_OP_12', 'Kanaal_PLAY4',
       'Kanaal_PLAY5', 'Kanaal_PLAY6', 'Kanaal_PLAY_SPORTS_OPEN', 'Kanaal_Q2',
       'Kanaal_RTL-TVI', 'Kanaal_TF1', 'Kanaal_VIER', 'Kanaal_VIJF',
       'Kanaal_VITAYA', 'Kanaal_VRT_1', 'Kanaal_VRT_CANVAS', 'Kanaal_VTM',
       'Kanaal_VTM2', 'Kanaal_VTM3', 'Kanaal_VTM4', 'Kanaal_VTM_GOLD',
       'Kanaal_ZES', 'isFeestdag_0', 'isFeestdag_1', 'Weekdag_0', 'Weekdag_1',
  

Index(['Sneeuw', 'Weercode', 'isWeekend', 'KijkersLag1', 'KijkersLag2',
       'KijkersLag3', 'hour_0', 'hour_1', 'hour_2', 'hour_6', 'hour_7',
       'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12', 'hour_13',
       'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18', 'hour_19',
       'hour_20', 'hour_21', 'hour_22', 'hour_23', 'Kanaal_AB3',
       'Kanaal_CANVAS', 'Kanaal_CAZ', 'Kanaal_Canvas',
       'Kanaal_DAZN_PRO_LEAGUE_1_(NL)', 'Kanaal_EEN',
       'Kanaal_ELEVEN_PRO_LEAGUE_1_NL', 'Kanaal_EUROSPORT_1_(NL)',
       'Kanaal_KETNET', 'Kanaal_LA_UNE', 'Kanaal_OP_12', 'Kanaal_PLAY4',
       'Kanaal_PLAY5', 'Kanaal_PLAY6', 'Kanaal_PLAY_SPORTS_OPEN', 'Kanaal_Q2',
       'Kanaal_RTL-TVI', 'Kanaal_TF1', 'Kanaal_VIER', 'Kanaal_VIJF',
       'Kanaal_VITAYA', 'Kanaal_VRT_1', 'Kanaal_VRT_CANVAS', 'Kanaal_VTM',
       'Kanaal_VTM2', 'Kanaal_VTM3', 'Kanaal_VTM4', 'Kanaal_VTM_GOLD',
       'Kanaal_ZES', 'isFeestdag_0', 'isFeestdag_1', 'Weekdag_0', 'Weekdag_1',
       'Weekdag_2', 'Week

## Voorspelling maken

In [39]:
def voorspellingMaken(teVoorspellenData):
  # global histKijkcijfersWeer
  # data['dateDiff'] = pd.to_datetime(data['dateDiff'])

  # end_date = data['dateDiff'].max()
  # start_date = end_date - timedelta(weeks=3)
  # histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
  # histWeerdata = ophalenWeerData(start_date, end_date)

  # histWeerdataClean = cleanWeerData(histWeerdata)
  # histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
  # # Merge historische kijkcijfers en weerdata
  # histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
  # histKijkcijfersWeer.dropna(inplace=True)

  # # te voorspellen data en weerdata mergen
  # teVoorspellenClean = cleanKijkcijferData(data)
  # teVoorspellenClean = teVoorspellenClean.drop(columns=['minute'])
  # teVoorspellenData = pd.merge(teVoorspellenClean, histWeerdataClean, on=['date', 'hour'], how='left')
  # teVoorspellenData = teVoorspellenData.drop(columns=['datetime', 'minute'])
  
  prepared = voorbereiding(teVoorspellenData, histKijkcijfersWeer)

  scaler = StandardScaler()

  preprocessed_data = scaler.fit_transform(prepared)

  print(preprocessed_data.shape)
  print(preprocessed_data[:5])

  with open('./models/lightGBM.pkl', 'rb') as file:
      lightgbm = pickle.load(file)
  print("Model features:", lightgbm.feature_name_)

  # Print de kolommen van je testdata
  print("Test features:", list(preprocessed_data.columns) if hasattr(preprocessed_data, 'columns') else preprocessed_data.shape)
  train_features = set(lightgbm.feature_name_)
  test_features = set(prepared.columns)
  print("In test, niet in train:", test_features - train_features)
  print("In train, niet in test:", train_features - test_features)

  predictions = lightgbm.predict(preprocessed_data)
  
  return predictions

Prediction resultaten

In [40]:
prediction = voorspellingMaken(teVoorspellenData)
prediction

['VRT_1' 'VTM' 'VRT_CANVAS' 'PLAY4' 'VTM2' 'PLAY5'
 'DAZN_PRO_LEAGUE_1_(NL)']
Index(['FullDate', 'Kijkers', 'Sneeuw', 'Weercode', 'isWeekend', 'KijkersLag1',
       'KijkersLag2', 'KijkersLag3', 'hour_0', 'hour_1', 'hour_2', 'hour_6',
       'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12',
       'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18',
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'Kanaal_AB3',
       'Kanaal_CANVAS', 'Kanaal_CAZ', 'Kanaal_Canvas',
       'Kanaal_DAZN_PRO_LEAGUE_1_(NL)', 'Kanaal_EEN',
       'Kanaal_ELEVEN_PRO_LEAGUE_1_NL', 'Kanaal_EUROSPORT_1_(NL)',
       'Kanaal_KETNET', 'Kanaal_LA_UNE', 'Kanaal_OP_12', 'Kanaal_PLAY4',
       'Kanaal_PLAY5', 'Kanaal_PLAY6', 'Kanaal_PLAY_SPORTS_OPEN', 'Kanaal_Q2',
       'Kanaal_RTL-TVI', 'Kanaal_TF1', 'Kanaal_VIER', 'Kanaal_VIJF',
       'Kanaal_VITAYA', 'Kanaal_VRT_1', 'Kanaal_VRT_CANVAS', 'Kanaal_VTM',
       'Kanaal_VTM2', 'Kanaal_VTM3', 'Kanaal_VTM4', 'Kanaal_VTM_GOLD',
   

c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


array([276079.95949137, 253449.33897115, 253449.33897115, 160680.95679356,
       171970.35949343, 184210.23291874, 261621.2275759 , 261621.2275759 ,
       276079.95949137, 145575.68765881])

Examen bestand

In [69]:
teVoorspellen = testCSV('./data csv/TEST_kijkcijfers.csv')

teVoorspellen['dateDiff'] = pd.to_datetime(teVoorspellen['dateDiff'])

end_date = teVoorspellen['dateDiff'].max()
start_date = end_date - timedelta(weeks=3)
histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
histWeerdata = ophalenWeerData(start_date, end_date)
print(teVoorspellen.columns)

X_test = teVoorspellen

histWeerdataClean = cleanWeerData(histWeerdata)
histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
# Merge historische kijkcijfers en weerdata
histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
histKijkcijfersWeer.dropna(inplace=True)
# te voorspellen data en weerdata mergen
teVoorspellenClean = cleanKijkcijferData(teVoorspellen)
teVoorspellenClean = teVoorspellenClean.drop(columns=['minute'])
teVoorspellenData = pd.merge(teVoorspellenClean, histWeerdataClean, on=['date', 'hour'], how='left')
teVoorspellenData = teVoorspellenData.drop(columns=['datetime', 'minute'])
#print("Aantal rijen in histKijkcijferWeerDf:", len(histKijkcijfersWeer))
#print(histKijkcijfersWeer.head()) 

prediction = voorspellingMaken(teVoorspellenData)

resultaten = pd.DataFrame({
    'Predicted': prediction
})


resultaten.head(10)

Ophalen kijkcijfer-data van 2025-4-25 tot 2025-5-15
KijkcijferData opgehaald
Ophalen historische data
Ophalen forecast data
2025-05-16 00:00:00
Weerdata opgehaald
Index(['dateDiff', 'ranking', 'description', 'channel', 'startTime', 'rLength',
       'live'],
      dtype='object')
['VRT_1' 'VTM' 'VRT_CANVAS' 'PLAY4' 'VTM2' 'PLAY5'
 'DAZN_PRO_LEAGUE_1_(NL)']
Index(['FullDate', 'Kijkers', 'Sneeuw', 'Weercode', 'isWeekend', 'KijkersLag1',
       'KijkersLag2', 'KijkersLag3', 'hour_0', 'hour_1', 'hour_2', 'hour_6',
       'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12',
       'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18',
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'Kanaal_AB3',
       'Kanaal_CANVAS', 'Kanaal_CAZ', 'Kanaal_Canvas',
       'Kanaal_DAZN_PRO_LEAGUE_1_(NL)', 'Kanaal_EEN',
       'Kanaal_ELEVEN_PRO_LEAGUE_1_NL', 'Kanaal_EUROSPORT_1_(NL)',
       'Kanaal_KETNET', 'Kanaal_LA_UNE', 'Kanaal_OP_12', 'Kanaal_PLAY4',
       'Kanaal_P

c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Predicted
0,276079.959491
1,251241.318160
2,251241.318160
3,161407.744307
4,172697.147007
5,184210.232919
6,260416.647368
7,260416.647368
8,276079.959491
9,147298.213985


Zelf testen

In [66]:
teVoorspellen = testing('./data csv/TEST_kijkcijfers.csv')

teVoorspellen['dateDiff'] = pd.to_datetime(teVoorspellen['dateDiff'])

end_date = teVoorspellen['dateDiff'].max()
start_date = end_date - timedelta(weeks=3)
histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
histWeerdata = ophalenWeerData(start_date, end_date)
print(teVoorspellen.columns)

X_test = teVoorspellen.drop(columns=['rateInK'])
y_test = teVoorspellen['rateInK'].apply(lambda x: int(''.join(str(x).split('.'))))

histWeerdataClean = cleanWeerData(histWeerdata)
histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
# Merge historische kijkcijfers en weerdata
histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
histKijkcijfersWeer.dropna(inplace=True)
# te voorspellen data en weerdata mergen
teVoorspellenClean = cleanKijkcijferData(teVoorspellen)
teVoorspellenClean = teVoorspellenClean.drop(columns=['minute'])
teVoorspellenData = pd.merge(teVoorspellenClean, histWeerdataClean, on=['date', 'hour'], how='left')
teVoorspellenData = teVoorspellenData.drop(columns=['datetime', 'minute'])
#print("Aantal rijen in histKijkcijferWeerDf:", len(histKijkcijfersWeer))
#print(histKijkcijfersWeer.head()) 

prediction = voorspellingMaken(teVoorspellenData)

# mae en mape berekenen
mae = mean_absolute_error(y_test, prediction)
mape = mean_absolute_percentage_error(y_test, prediction)
print(f"MAE: {mae}")
print(f"MAPE: {mape}")

resultaten = pd.DataFrame({
    'Actual': y_test,
    'Predicted': prediction
})

resultaten['Difference'] = resultaten['Actual'] - resultaten['Predicted']

resultaten.head(10)

Ophalen kijkcijfer-data van 2025-4-25 tot 2025-5-15
KijkcijferData opgehaald
Ophalen historische data
Ophalen forecast data
2025-05-16 00:00:00
Weerdata opgehaald
Index(['dateDiff', 'ranking', 'description', 'channel', 'startTime', 'rLength',
       'rateInK', 'live'],
      dtype='object')
['VRT_1' 'VTM' 'VRT_CANVAS' 'PLAY4' 'VTM2' 'PLAY5'
 'DAZN_PRO_LEAGUE_1_(NL)']
Index(['FullDate', 'Kijkers', 'Sneeuw', 'Weercode', 'isWeekend', 'KijkersLag1',
       'KijkersLag2', 'KijkersLag3', 'hour_0', 'hour_1', 'hour_2', 'hour_6',
       'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12',
       'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18',
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'Kanaal_AB3',
       'Kanaal_CANVAS', 'Kanaal_CAZ', 'Kanaal_Canvas',
       'Kanaal_DAZN_PRO_LEAGUE_1_(NL)', 'Kanaal_EEN',
       'Kanaal_ELEVEN_PRO_LEAGUE_1_NL', 'Kanaal_EUROSPORT_1_(NL)',
       'Kanaal_KETNET', 'Kanaal_LA_UNE', 'Kanaal_OP_12', 'Kanaal_PLAY4',
     

c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Actual,Predicted,Difference
0,893239,276079.959491,617159.040509
1,823019,251241.318160,571777.681840
2,997337,251241.318160,746095.681840
3,501293,161407.744307,339885.255693
4,527109,172697.147007,354411.852993
5,620036,184210.232919,435825.767081
6,521098,260416.647368,260681.352632
7,428309,260416.647368,167892.352632
8,65023,276079.959491,-211056.959491
9,398209,147298.213985,250910.786015


In [67]:
resultaten.describe()

,Actual,Predicted,Difference
count,10.000000,10.000000,10.000000
mean,577467.200000,224108.918826,353358.281174
std,272153.335173,51182.909159,268859.884337
min,65023.000000,147298.213985,-211056.959491
25%,446555.000000,175575.418485,253353.427669
50%,524103.500000,251241.318160,347148.554343
75%,772273.250000,260416.647368,537789.703150
max,997337.000000,276079.959491,746095.681840
